# Can You Beat the Closing Line?
Walkthrough of the analysis. Logic lives in `src/clv.py` and `sql/closing_line.sql`.

In [1]:
import sys; sys.path.insert(0,'../src')
from clv import *
con=connect(); g=con.execute('SELECT * FROM games').df(); len(g)

7354

## 1. Efficiency: regress outcomes on closing lines (efficient = slope 1, intercept 0)

In [2]:
efficiency(g)

{'result': {'intercept': np.float64(-0.016343645859609273),
  'slope': np.float64(1.041894017240638),
  'slope_ci': (0.9913222666660124, 1.0924657678152634),
  'mean_err': np.float64(0.07771280935545281),
  'mae': np.float64(10.265501767745445),
  'n': 7354},
 'total': {'intercept': np.float64(3.633485434145171),
  'slope': np.float64(0.9328924702122383),
  'slope_ci': (0.868740001461349, 0.9970449389631276),
  'mean_err': np.float64(0.7097497960293717),
  'mae': np.float64(10.61837095458254),
  'n': 7354}}

## 2. Moneyline calibration after vig removal

In [3]:
d,cal,brier=moneyline_calibration(g); print(brier, d.overround.mean()); cal

0.21134138630865953 0.027457857585823222


/workspace/portfolio/nfl-closing-line/notebooks/../src/clv.py:21: RuntimeWarning: divide by zero encountered in divide
  return np.where(ml < 0, -ml / (-ml + 100), 100 / (ml + 100))
/workspace/portfolio/nfl-closing-line/notebooks/../src/clv.py:21: RuntimeWarning: divide by zero encountered in divide
  return np.where(ml < 0, -ml / (-ml + 100), 100 / (ml + 100))


,n,pred,actual
bin,,,
"(0.0, 0.1]",7,0.092939,0.000000
"(0.1, 0.2]",108,0.162703,0.194444
"(0.2, 0.3]",384,0.256445,0.255208
"(0.3, 0.4]",672,0.353026,0.343750
"(0.4, 0.5]",756,0.447898,0.437831
"(0.5, 0.6]",856,0.556523,0.532710
"(0.6, 0.7]",1107,0.648434,0.617886
"(0.7, 0.8]",971,0.748444,0.759011
"(0.8, 0.9]",455,0.842496,0.861538


## 3. Candidate biases: binomial tests, Wilson CIs, Holm correction

In [4]:
test_strategies(con.execute('SELECT * FROM strategy_summary').df()).round(4)

,strategy,games,bets,wins,losses,win_rate,roi_per_unit,p_vs_50,p_beats_vig,ci_low,ci_high,p_vs_50_holm,p_beats_vig_holm,sig_bias_holm
0,"Bet unders, wind 15+ mph",684,674,384.0,290.0,0.5697,0.0877,0.0003,0.0093,0.5321,0.6066,0.0033,0.0932,True
1,Bet playoff underdogs ATS,308,302,165.0,137.0,0.5464,0.0430,0.1201,0.2338,0.4900,0.6016,0.6641,1.0000,False
2,Bet divisional underdogs ATS,2748,2673,1396.0,1277.0,0.5223,-0.0030,0.0225,0.5715,0.5033,0.5412,0.2021,1.0000,False
3,Bet underdogs of 7+ ATS,2221,2161,1123.0,1038.0,0.5197,-0.0079,0.0707,0.6581,0.4986,0.5407,0.4952,1.0000,False
4,Bet team with rest edge ATS,1426,1394,709.0,685.0,0.5086,-0.0290,0.5379,0.8776,0.4824,0.5348,1.0000,1.0000,False
5,Bet home underdogs ATS,2486,2417,1221.0,1196.0,0.5052,-0.0356,0.6254,0.9682,0.4852,0.5251,1.0000,1.0000,False
6,Bet all overs,7354,7248,3584.0,3664.0,0.4945,-0.0560,0.3534,1.0000,0.4830,0.5060,1.0000,1.0000,False
7,Bet home team ATS,7256,7057,3461.0,3596.0,0.4904,-0.0637,0.1107,1.0000,0.4788,0.5021,0.6641,1.0000,False
8,Bet primetime overs,1255,1233,601.0,632.0,0.4874,-0.0695,0.3929,0.9951,0.4596,0.5153,1.0000,1.0000,False
9,Bet favorites ATS,7323,7122,3469.0,3653.0,0.4871,-0.0701,0.0301,1.0000,0.4755,0.4987,0.2409,1.0000,False


## 4. Walk-forward: choose in-sample (<2015), evaluate 2015+

In [5]:
walk_forward(con).round(4)

,in_sample_rate,oos_rate,oos_bets,picked,oos_roi
strategy,,,,,
"Bet unders, wind 15+ mph",0.5724,0.5644,225,True,0.0776
Bet playoff underdogs ATS,0.5380,0.5573,131,True,0.0638
Bet underdogs of 7+ ATS,0.5215,0.5169,857,False,-0.0132
Bet divisional underdogs ATS,0.5197,0.5261,1072,False,0.0044
Bet primetime overs,0.5111,0.4627,603,False,-0.1167
Bet home underdogs ATS,0.5027,0.5082,1102,False,-0.0299
Bet all overs,0.4988,0.4886,3078,False,-0.0672
Bet team with rest edge ATS,0.4979,0.5204,663,False,-0.0066
Bet home team ATS,0.4912,0.4894,2961,False,-0.0658
